# AUTOMATED TRANSACTION REPORT GENERATION SYSTEM

### *Splits a master CSV transaction file into separate Excel files by merchant, then applies professional formatting to each output file.*

In [1]:
# ------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------
import pandas as pd
import openpyxl
import os
from copy import copy
from datetime import datetime

# ------------------------------------------
# CONFIGURATION - UPDATE THESE PATHS
# ------------------------------------------
SOURCE_FILE_PATH = r'C:\Users\ARJUN\Downloads\TRANSACTIONS.csv'
SPLIT_BY_COLUMN = 'MERCHANT'
OUTPUT_DIRECTORY = r'C:\Users\ARJUN\Downloads\destination_folder'

# ------------------------------------------
# STYLE DEFINITIONS
# ------------------------------------------
THIN_BORDER_STYLE = openpyxl.styles.Border(
    left = openpyxl.styles.Side(style = 'thin'),
    right = openpyxl.styles.Side(style = 'thin'),
    top = openpyxl.styles.Side(style = 'thin'),
    bottom = openpyxl.styles.Side(style = 'thin')
    )

CENTER_ALIGNMENT = openpyxl.styles.Alignment(
    horizontal = 'center',
    vertical = 'center'
)

# ------------------------------------------
# LOAD SOURCE DATA
# ------------------------------------------
source_dataframe = pd.read_csv(SOURCE_FILE_PATH)

# ------------------------------------------
# FILTER AND EXPORT DATA IN EXCEL AS PER THE SPLIT COLUMN
# ------------------------------------------
for merchant_name in source_dataframe['MERCHANT'].unique():
    filtered_data = source_dataframe[source_dataframe['MERCHANT'] == merchant_name]
    
    # Build the output path (file path + file name)
    output_file_path = os.path.join(OUTPUT_DIRECTORY, f'SUSPICIOUS_TRANSACTIONS_REPORT - {merchant_name}.xlsx')
    
    # Save the filtered data (initial unformatted version)
    filtered_data.to_excel(output_file_path, index = False)
    
    # ------------------------------------------
    # LOAD WORKBOOK FOR FORMATTING
    # ------------------------------------------
    workbook = openpyxl.load_workbook(output_file_path)
    active_worksheet = workbook.active
    
    # ------------------------------------------
    # FORMAT EACH CELL
    # ------------------------------------------
    for row_cells in active_worksheet.iter_rows():
        for individual_row_cell in row_cells:
            
            # Make header row (row 1) bold
            if individual_row_cell.row == 1:
                bold_header_font = copy(individual_row_cell.font) # font objects are immutable hence, copy()
                bold_header_font.bold = True
                individual_row_cell.font = bold_header_font
                
            # Apply thin borders
            individual_row_cell.border = THIN_BORDER_STYLE
            
            # Apply center alignment
            individual_row_cell.alignment = CENTER_ALIGNMENT
            
            # Format datetime values
            if isinstance(individual_row_cell.value, datetime):
                individual_row_cell.number_format = "DD/MM/YYYY HH:MM"
                
    # ------------------------------------------
    # AUTO-FIT COLUMN WIDTHS
    # ------------------------------------------
    for column_cells in active_worksheet.columns:
        # Get column letter (A, B, C etc.)
        first_cell_in_column = column_cells[0]
        column_letter = first_cell_in_column.column_letter
        
        # Find the longest text in this column by initializing a counter
        max_text_length = 0
        
        for individual_column_cell in column_cells:
            # Only process non empty cells
            if individual_column_cell.value:
                current_text_length = len(str(individual_column_cell.value))
                if current_text_length > max_text_length:
                    max_text_length = current_text_length
                    
        # Set column width with additional padding
        if max_text_length > 0:
            active_worksheet.column_dimensions[column_letter].width = max_text_length + 4
            
    # Save the formatted workbook
    workbook.save(output_file_path)
    
print('✅ PROCESSING COMPLETE')
print(f'📁 OUTPUT DIRECTORY: {OUTPUT_DIRECTORY}')

✅ PROCESSING COMPLETE
📁 OUTPUT DIRECTORY: C:\Users\ARJUN\Downloads\destination_folder
